# Libaries

In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from src.benchmark import (
    BenchmarkConfig,
    PairDatasetBuilder,
    PairwiseElasticityPipeline,
    BootstrapSummarizer,
)
from src.dominick import DominickDataLoader
from src.utils import TemporalSplitter, BlockBootstrapSampler

In [6]:
TRAIN_FRAC = 0.8
#N_FOLDS = 5
N_FOLDS = 1
#N_BOOTSTRAP = 20
N_BOOTSTRAP = 1
SELECTED_UPCS = [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]
config = BenchmarkConfig()

# Loader

In [7]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")
df = df[df["upc_code"].isin(SELECTED_UPCS)].copy()

pair_builder = PairDatasetBuilder(control_cols=config.control_cols)
pair_df = pair_builder.build(df)

print(f"Dataset: {df.shape}, Pairs: {pair_df.shape}")

Dataset: (73209, 44), Pairs: (225004, 24)


# K-fold

In [8]:
splitter = TemporalSplitter(week_col="week_id")
pipeline = PairwiseElasticityPipeline(config)

fold_results = []
for fold_idx, (train_fold, val_fold) in enumerate(splitter.expanding_splits(pair_df, N_FOLDS)):
    print(f"Fold {fold_idx} of {N_FOLDS}")
    res = pipeline.run(train_fold, val_fold)
    res["fold"] = fold_idx
    fold_results.append(res)

all_folds = pd.concat(fold_results, ignore_index=True)
ok_folds = all_folds[all_folds["status"] == "ok"].copy()
print(f"K-fold raw: {len(ok_folds)} rows")

Fold 0 of 1
K-fold raw: 938 rows


# Bootstrap

In [9]:
train_df, val_df = splitter.single_split(pair_df, train_frac=TRAIN_FRAC)
train_weeks = sorted(train_df["week_id"].unique())

sampler = BlockBootstrapSampler(week_col="week_id", block_size=4, rng=np.random.default_rng(42))

bootstrap_results = []
for b in range(N_BOOTSTRAP):
    print(f"Bootstrap {b} of {N_BOOTSTRAP}")
    train_bs = sampler.sample(train_df, train_weeks)
    res = pipeline.run(train_bs, val_df)
    res["bootstrap_run"] = b
    bootstrap_results.append(res)

all_bootstrap = pd.concat(bootstrap_results, ignore_index=True)
ok_bs = all_bootstrap[all_bootstrap["status"] == "ok"].copy()
print(f"Bootstrap raw: {len(ok_bs)} rows")

Bootstrap 0 of 1
Bootstrap raw: 544 rows


# Summaries

In [ ]:
summarizer = BootstrapSummarizer()
bootstrap_summary = summarizer.summarize_raw(ok_bs)

print(f"K-fold raw: {len(ok_folds)}")
print(f"Bootstrap summary: {len(bootstrap_summary)}")

K-fold raw: 938
Bootstrap summary: 544


# Display

In [11]:
display(ok_folds.head(10))
display(ok_folds[["own_elasticity", "cross_elasticity", "mae_val", "rmse_val", "r2_val"]].describe())
display(bootstrap_summary.head(10))
display(all_folds["status"].value_counts(dropna=False))
display(all_bootstrap["status"].value_counts(dropna=False))

,store_code,pair_id,upc_i,upc_j,status,n_train,n_val,own_elasticity,own_elasticity_ci_low,own_elasticity_ci_high,own_elasticity_p_value,cross_elasticity,cross_elasticity_ci_low,cross_elasticity_ci_high,cross_elasticity_p_value,mae_val,rmse_val,r2_val,fold
0,5,3410017306.0__7289000011.0,3410017306,7289000011,ok,49,56,-3.231876,-10.292528,3.828776,3.696468e-01,0.143843,-16.414304,16.701989,0.986416,0.502538,0.588763,0.036916,0
1,5,3410017306.0__7289000011.0,7289000011,3410017306,ok,49,56,1.428486,-10.277905,13.134876,8.109762e-01,-3.832222,-9.843352,2.178908,0.211476,0.839168,1.011244,-1.902501,0
2,8,1820000784.0__3410010505.0,1820000784,3410010505,ok,140,151,-3.980905,-5.830347,-2.131462,2.456042e-05,-1.234048,-2.646545,0.178449,0.086832,0.536723,0.695363,0.104700,0
3,8,1820000784.0__3410010505.0,3410010505,1820000784,ok,140,151,-4.212563,-6.343453,-2.081673,1.067742e-04,-0.775208,-2.015348,0.464932,0.220512,0.508660,0.660931,-0.081145,0
4,8,1820000784.0__3410017306.0,1820000784,3410017306,ok,142,69,-4.399785,-6.211706,-2.587863,1.942971e-06,-0.111635,-2.372310,2.149041,0.922897,0.573595,0.709691,0.242645,0
5,8,1820000784.0__3410017306.0,3410017306,1820000784,ok,142,69,-6.894701,-8.926355,-4.863047,2.902981e-11,-0.363335,-1.464546,0.737876,0.517843,1.022234,1.159344,-3.894083,0
6,8,1820000784.0__7289000011.0,1820000784,7289000011,ok,101,140,-3.765933,-5.760757,-1.771109,2.154953e-04,-3.926900,-10.769218,2.915418,0.260653,0.809934,0.961734,-0.731025,0
7,8,1820000784.0__7289000011.0,7289000011,1820000784,ok,101,140,-2.149943,-9.852645,5.552759,5.843402e-01,-0.882123,-2.169530,0.405285,0.179287,1.360642,1.704132,-4.821465,0
8,8,3410010505.0__3410017306.0,3410010505,3410017306,ok,141,69,-4.753681,-7.001935,-2.505427,3.411155e-05,0.164301,-1.859324,2.187927,0.873564,0.584045,0.737779,-0.505693,0
9,8,3410010505.0__3410017306.0,3410017306,3410010505,ok,141,69,-6.853583,-8.891930,-4.815236,4.397279e-11,-1.169837,-2.322327,-0.017348,0.046650,0.698215,0.804725,-1.357991,0


,own_elasticity,cross_elasticity,mae_val,rmse_val,r2_val
count,938.000000,938.000000,938.000000,938.000000,938.000000
mean,-4.431440,-0.234030,0.938339,1.108656,-3.032490
std,3.125673,2.195133,0.712547,0.780058,8.240392
min,-13.428422,-12.408779,0.226755,0.301232,-119.899110
25%,-6.031474,-0.899938,0.532715,0.666100,-3.153279
50%,-4.765493,-0.289291,0.734035,0.914975,-0.893684
75%,-3.510824,0.346783,1.105265,1.301732,-0.016822
max,13.982406,15.694139,7.748612,9.405785,0.759215


,store_code,pair_id,upc_i,upc_j,own_elasticity_mean,own_elasticity_std,own_elasticity_ci_low,own_elasticity_ci_high,cross_elasticity_mean,cross_elasticity_std,cross_elasticity_ci_low,cross_elasticity_ci_high,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std
0,8,1820000784.0__3410010505.0,1820000784,3410010505,-4.780789,NaN,-4.780789,-4.780789,0.087502,NaN,0.087502,0.087502,0.508481,NaN,0.676672,NaN,0.051983,NaN
1,8,1820000784.0__3410010505.0,3410010505,1820000784,-3.141082,NaN,-3.141082,-3.141082,1.020363,NaN,1.020363,1.020363,0.511590,NaN,0.644569,NaN,-0.321729,NaN
2,8,1820000784.0__7289000011.0,1820000784,7289000011,-3.949830,NaN,-3.949830,-3.949830,-5.621814,NaN,-5.621814,-5.621814,0.661302,NaN,0.847643,NaN,-0.487598,NaN
3,8,1820000784.0__7289000011.0,7289000011,1820000784,0.581050,NaN,0.581050,0.581050,-0.548107,NaN,-0.548107,-0.548107,0.723580,NaN,0.882254,NaN,-0.651729,NaN
4,8,3410010505.0__7289000011.0,3410010505,7289000011,-2.666102,NaN,-2.666102,-2.666102,-2.648494,NaN,-2.648494,-2.648494,0.532577,NaN,0.640713,NaN,-0.305963,NaN
5,8,3410010505.0__7289000011.0,7289000011,3410010505,0.725548,NaN,0.725548,0.725548,0.194114,NaN,0.194114,0.194114,0.649290,NaN,0.840092,NaN,-0.497631,NaN
6,9,1820000784.0__3410010505.0,1820000784,3410010505,-4.736355,NaN,-4.736355,-4.736355,0.990426,NaN,0.990426,0.990426,0.848881,NaN,1.039104,NaN,-1.408668,NaN
7,9,1820000784.0__3410010505.0,3410010505,1820000784,-2.531342,NaN,-2.531342,-2.531342,2.257072,NaN,2.257072,2.257072,0.409080,NaN,0.512888,NaN,-0.325425,NaN
8,9,1820000784.0__7289000011.0,1820000784,7289000011,-4.388069,NaN,-4.388069,-4.388069,0.211044,NaN,0.211044,0.211044,0.719903,NaN,0.906741,NaN,-0.834112,NaN
9,9,1820000784.0__7289000011.0,7289000011,1820000784,-1.309715,NaN,-1.309715,-1.309715,0.906114,NaN,0.906114,0.906114,0.424171,NaN,0.540719,NaN,0.312925,NaN


status
ok    938
Name: count, dtype: int64

status
ok    544
Name: count, dtype: int64

# Save

In [ ]:
ok_folds.to_csv("../data/benchmark_kfold_raw.csv", index=False)
ok_bs.to_csv("../data/benchmark_bootstrap_raw.csv", index=False)
bootstrap_summary.to_csv("../data/benchmark_elasticities_bootstrap_summary.csv", index=False)
print("Saved OK")

Saved OK
